# Classification — train/test split + cross-validation

**ML Teach by Doing · Day 08 · companion notebook 2 of 2**

Same data, same algorithm as notebook 1 — but this time we grade honestly:

1. **Split** the 20 animals: 16 for training, 4 locked in a test vault (`train_test_split`)
2. Train on the 16, then open the vault **once** for a fair test error
3. Use **5-fold cross-validation** (`KFold`) to choose the hyperparameter `k` — without ever touching the vault

New ingredient: **scikit-learn**, which provides `train_test_split` and `KFold` ready-made.

> ⚠️ Run cells **top to bottom** — the seed only guarantees identical results for an identical sequence of random draws.

In [ ]:
# ============================================================
# SETUP
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold   # ready-made ML utilities

In [ ]:
np.random.seed(42)   # reproducibility, same as notebook 1

In [ ]:
# ============================================================
# STEP 1 — manufacture the data (identical to notebook 1)
# ============================================================
N = 10  # animals per species

dog_whisker_length = np.random.normal(5, 1, N)   # (10,)
dog_ear_flappiness = np.random.normal(8, 1, N)   # (10,)
cat_whisker_length = np.random.normal(8, 1, N)   # (10,)
cat_ear_flappiness = np.random.normal(5, 1, N)   # (10,)

## Packing the data the scikit-learn way

scikit-learn wants **rows = examples**. So this notebook uses `.T` to flip each species into a `(10, 2)` block, stacks them into one `X` of shape `(20, 2)`, and adds a label vector `y`:

- label **0 = dog**
- label **1 = cat**

(Notebook 1 used columns-as-animals for `np.dot`; here rows-as-animals for sklearn. We'll flip back with `.T` when it's time to compute scores.)

In [ ]:
# ============================================================
# PACKING — X: one animal per ROW, y: its label
# ============================================================
dogs = np.vstack((dog_whisker_length, dog_ear_flappiness)).T   # (10, 2) — row = one dog
cats = np.vstack((cat_whisker_length, cat_ear_flappiness)).T   # (10, 2)

X = np.vstack((dogs, cats))                     # (20, 2) — all animals
y = np.concatenate((np.zeros(N), np.ones(N)))   # (20,)   — 0 = dog, 1 = cat

print("X:", X.shape, "| y:", y.shape)

## The split — 16 for training, 4 for the vault

`test_size=0.2` reserves 20% for testing. `random_state=42` is sklearn's own seed — same split every run.

In [ ]:
# ============================================================
# THE SPLIT — carve out the test vault
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("X_train:", X_train.shape, " -> 16 animals for training")
print("X_test: ", X_test.shape, " ->  4 animals locked in the vault")
print("y_test:", y_test, "(0 = dog, 1 = cat)")

In [ ]:
# Visualise the split: squares are the vault animals.
plt.scatter(X_train[y_train == 0][:, 0], X_train[y_train == 0][:, 1], label="dogs — train")
plt.scatter(X_train[y_train == 1][:, 0], X_train[y_train == 1][:, 1], label="cats — train")
plt.scatter(X_test[y_test == 0][:, 0], X_test[y_test == 0][:, 1],
            c="darkblue", marker="s", s=80, label="dogs — TEST")
plt.scatter(X_test[y_test == 1][:, 0], X_test[y_test == 1][:, 1],
            c="darkred", marker="s", s=80, label="cats — TEST")
plt.xlabel("whisker length  (x1)")
plt.ylabel("ear flappiness index  (x2)")
plt.title("16 training animals + 4 vault animals")
plt.legend()
plt.show()

In [ ]:
# ============================================================
# THE ALGORITHM — identical to notebook 1
# ============================================================
def compute_error(dogs, cats, theta, theta_0):
    # dogs, cats: (2, n) — each COLUMN is one animal
    dog_scores = np.dot(theta, dogs) + theta_0   # (n,)
    cat_scores = np.dot(theta, cats) + theta_0   # (n,)
    return np.sum(dog_scores < 0) + np.sum(cat_scores > 0)


def random_linear_classifier(dogs, cats, k, d):
    best_error, best_theta, best_theta_0 = np.inf, None, None
    for _ in range(k):
        theta   = np.random.normal(size=d)
        theta_0 = np.random.normal()
        error = compute_error(dogs, cats, theta, theta_0)
        if error < best_error:
            best_error, best_theta, best_theta_0 = error, theta, theta_0
    return best_theta, best_theta_0, best_error

## Train on the 16 only

Unpack the training animals back into dogs and cats (using the labels), and transpose back to `(2, n)` columns — the layout `np.dot` wants.

In [ ]:
# ============================================================
# TRAIN — the algorithm only ever sees the 16
# ============================================================
train_dogs = X_train[y_train == 0].T   # (2, n_dogs) — back to columns for np.dot
train_cats = X_train[y_train == 1].T   # (2, n_cats)
print("train_dogs:", train_dogs.shape, "| train_cats:", train_cats.shape)

best_theta, best_theta_0, _ = random_linear_classifier(train_dogs, train_cats, k=100, d=2)
train_error = compute_error(train_dogs, train_cats, best_theta, best_theta_0)
print("training error:", train_error, "/", X_train.shape[0])

## The vault opens — once

Grade the finished classifier on the 4 animals it has never seen.

In [ ]:
# ============================================================
# TEST — one honest measurement
# ============================================================
test_dogs = X_test[y_test == 0].T   # (2, n)
test_cats = X_test[y_test == 1].T
test_error = compute_error(test_dogs, test_cats, best_theta, best_theta_0)
print("test error:", test_error, "/", X_test.shape[0])

In [ ]:
# Truth vs prediction on the vault animals.
# Circles = actual species. Crosses = what the classifier says.
# A cross inside every circle = perfect predictions.
test_scores = np.dot(best_theta, X_test.T) + best_theta_0   # (4,)
predictions = np.sign(test_scores)                          # +1 = "dog", -1 = "cat"
print("scores:     ", np.round(test_scores, 2))
print("predictions:", predictions, "  (+1 dog / -1 cat)")
print("truth:      ", np.where(y_test == 0, 1.0, -1.0))

plt.scatter(X_test[y_test == 0][:, 0], X_test[y_test == 0][:, 1],
            s=200, facecolors="none", edgecolors="tab:blue", label="actual dogs")
plt.scatter(X_test[y_test == 1][:, 0], X_test[y_test == 1][:, 1],
            s=200, facecolors="none", edgecolors="tab:orange", label="actual cats")
plt.scatter(X_test[predictions == 1][:, 0], X_test[predictions == 1][:, 1],
            marker="x", c="tab:blue", label="predicted dog")
plt.scatter(X_test[predictions == -1][:, 0], X_test[predictions == -1][:, 1],
            marker="x", c="tab:orange", label="predicted cat")
plt.xlabel("whisker length  (x1)")
plt.ylabel("ear flappiness index  (x2)")
plt.title("Vault animals: truth (circles) vs prediction (crosses)")
plt.legend()
plt.show()

## Cross-validation — choosing `k` fairly

We picked `k=100` out of thin air. Is it a *good* setting? We're not allowed to compare candidates on the vault (that would tune on the test set), so we recycle the **training** data:

- Chop the 16 training animals into **5 folds** (`KFold`)
- For each candidate `k`: withhold one fold, train on the rest, measure mistakes on the withheld fold; rotate through all 5 folds; **average** the 5 errors
- Crown the `k` with the lowest average

This grades the *recipe* (the algorithm + its setting), not any single line.

In [ ]:
# ============================================================
# CROSS-VALIDATION — grade the recipe, not a line
# ============================================================
def cross_validate(X, y, k_values, n_splits=5):
    kf = KFold(n_splits=n_splits)                   # chops the data into 5 folds
    avg_errors = []
    for k in k_values:                              # audition each candidate k...
        fold_errors = []
        for train_idx, val_idx in kf.split(X):      # ...over 5 rounds of hide-and-train
            X_tr, y_tr = X[train_idx], y[train_idx]
            X_va, y_va = X[val_idx], y[val_idx]
            th, th0, _ = random_linear_classifier(
                X_tr[y_tr == 0].T, X_tr[y_tr == 1].T, k, d=2)
            fold_errors.append(compute_error(
                X_va[y_va == 0].T, X_va[y_va == 1].T, th, th0))
        avg_errors.append(np.mean(fold_errors))     # the recipe's grade for this k
    best_k = k_values[int(np.argmin(avg_errors))]   # argmin -> INDEX of the smallest
    return best_k, avg_errors


k_values = [1, 10, 50, 100, 200]
best_k, avg_errors = cross_validate(X_train, y_train, k_values)   # TRAIN data only!

print("k_values:  ", k_values)
print("avg_errors:", [round(float(e), 2) for e in avg_errors])
print("best_k:", best_k)

**Expected with seed 42:** `avg_errors ≈ [1.6, 0.8, 0.0, 0.0, 0.2]` → `best_k = 50`.

Two things worth staring at:

1. `k=50` and `k=100` **tie** at 0.0 — and `np.argmin` returns the index of the *first* minimum, so the cheaper `k=50` wins. A tie-break you got for free.
2. The lecture's run crowned `k=200`. Different seed, different verdict — cross-validation grades are *estimates with noise*, not gospel. On an easily-separable dataset like this one, modest `k` is plenty.

And notice: `X_test` appears nowhere in that cell. The vault stayed shut.

In [ ]:
# ============================================================
# FINAL RUN — retrain with the winning k, then grade
# ============================================================
best_theta, best_theta_0, _ = random_linear_classifier(train_dogs, train_cats, k=best_k, d=2)

final_train_error = compute_error(train_dogs, train_cats, best_theta, best_theta_0)
final_test_error  = compute_error(test_dogs, test_cats, best_theta, best_theta_0)
print("final training error:", final_train_error, "/ 16")
print("final test error:    ", final_test_error, "/ 4")

# The final picture: training dots, vault squares, and the chosen boundary.
x1 = np.linspace(2, 11, 100)
x2 = -(best_theta[0] * x1 + best_theta_0) / best_theta[1]

plt.scatter(X_train[y_train == 0][:, 0], X_train[y_train == 0][:, 1], label="dogs — train")
plt.scatter(X_train[y_train == 1][:, 0], X_train[y_train == 1][:, 1], label="cats — train")
plt.scatter(X_test[y_test == 0][:, 0], X_test[y_test == 0][:, 1],
            c="darkblue", marker="s", s=80, label="dogs — TEST")
plt.scatter(X_test[y_test == 1][:, 0], X_test[y_test == 1][:, 1],
            c="darkred", marker="s", s=80, label="cats — TEST")
plt.plot(x1, x2, "r--", label=f"boundary (k={best_k})")
plt.xlabel("whisker length  (x1)")
plt.ylabel("ear flappiness index  (x2)")
plt.title("Final classifier, chosen by cross-validation")
plt.ylim(2, 11)
plt.legend()
plt.show()

**Expected with seed 42:** final training error `1 / 16`, final test error `0 / 4`.

Wait — cross-validation crowned `k=50`, and now a `k=50` run makes a training mistake that the earlier `k=100` run didn't?

Welcome to random search. Every run is a **fresh batch of lottery tickets**: by the time this cell runs, the random number generator has moved on, so these 50 guesses are different from any 50 seen before — and this batch happened to be slightly unlucky on one training point (while still acing the vault). This wobble is not a bug in your code; it *is* the algorithm. It's also exactly why the next chapter replaces blind guessing with something that learns from each guess: **gradient descent**.

## Experiments — break it on purpose

1. Add `350` to `k_values`, like the lecture did. Does the verdict change?
2. Change `n_splits` to 4 or 8. Does `best_k` move?
3. Fairer averaging: append `... / len(val_idx)` to use error *rate* per fold instead of raw counts (folds have 4,3,3,3,3 animals — should big folds count more?)
4. `KFold(n_splits=5, shuffle=True, random_state=0)` — does shuffling folds change anything here? Why might it matter on *sorted* data?
5. Set `test_size=0.5`. Better test estimate, worse training — where's the trade-off?
6. Remove `np.random.seed(42)` and run everything five times. Watch which numbers wobble and which stay put.